# University of Engineering and Technology Peshawar, Nowshera Campus
Name: MUNSIFF ALI

Registration Number: 22JZELE0489
## Lab 5: AEP Dataset - Normalization, One-Hot Encoding & Cyclic Features


Name: MUNSIFF ALI  
Registration Number:22JZELE0489 


---
**Objective:Perform data normalization using MinMaxScaler, apply One-Hot Encoding on categorical features, and generate cyclic features for time-related columns.

## 1. Import Libraries

In [2]:
pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement scikit-learn (from versions: none)
ERROR: No matching distribution found for scikit-learn


In [2]:
import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from timeseires.utils import t_v_t_split as sp

In [3]:
# pip install pickle

## 2. Load Dataset

In [4]:
df = pd.read_csv(r'C:\Users\Admin\Downloads\labs\New folder\5_features_extracted.csv', 
                 index_col=['Datetime'], parse_dates=['Datetime'])
df.head()

,aep,year_day,holiday,weekend,winter,spring,summer,fall,hour,month,day_of_week
Datetime,,,,,,,,,,,
2004-10-01 01:00:00,12379.0,275,0,0,0,0,0,1,1,10,4
2004-10-01 02:00:00,11935.0,275,0,0,0,0,0,1,2,10,4
2004-10-01 03:00:00,11692.0,275,0,0,0,0,0,1,3,10,4
2004-10-01 04:00:00,11597.0,275,0,0,0,0,0,1,4,10,4
2004-10-01 05:00:00,11681.0,275,0,0,0,0,0,1,5,10,4


## 3. Dataset Information

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 121296 entries, 2004-10-01 01:00:00 to 2018-08-03 00:00:00
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   aep          121296 non-null  float64
 1   year_day     121296 non-null  int64  
 2   holiday      121296 non-null  int64  
 3   weekend      121296 non-null  int64  
 4   winter       121296 non-null  int64  
 5   spring       121296 non-null  int64  
 6   summer       121296 non-null  int64  
 7   fall         121296 non-null  int64  
 8   hour         121296 non-null  int64  
 9   month        121296 non-null  int64  
 10  day_of_week  121296 non-null  int64  
dtypes: float64(1), int64(10)
memory usage: 11.1 MB


## 4. Train-Validation-Test Split

In [6]:
train_set, validation_set, test_set = sp.t_v_t(df, 70, 20, 10)
print(train_set.shape)
print(validation_set.shape)
print(test_set.shape)

(84907, 11)
(24259, 11)
(12130, 11)


## 5. Normalization (MinMaxScaler)

In [7]:
train_set_load = train_set['aep'].values.reshape(-1, 1)
validation_set_load = validation_set['aep'].values.reshape(-1, 1)
test_set_load = test_set['aep'].values.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_set_load)

scaled_train = scaler.transform(train_set_load)
scaled_val = scaler.transform(validation_set_load)
scaled_test = scaler.transform(test_set_load)

pickle.dump(scaler, open("AEPscaler.pkl", "wb"))
print(scaled_train.shape)

(84907, 1)


## 6. One-Hot Encoding

In [8]:
train_setO = train_set.values
holiday = train_setO[:, 2:3]
weekend = train_setO[:, 3:4]

en_holiday = OneHotEncoder(handle_unknown='ignore')
en_weekend = OneHotEncoder(handle_unknown='ignore')

holidayt = en_holiday.fit_transform(holiday).toarray()
weekendt = en_weekend.fit_transform(weekend).toarray()

train_categorical = np.concatenate((holidayt, weekendt), axis=1)
print(train_categorical.shape)

(84907, 4)


## 7. Cyclic Features

In [9]:
cyclic_train = train_set[['month','day_of_week','hour','winter','spring','summer','fall','year_day']].values

def cyclic_encode(data, period):
    return np.sin(2*np.pi*data/period), np.cos(2*np.pi*data/period)

sin_month, cos_month = cyclic_encode(cyclic_train[:,0:1], 12)
sin_week, cos_week = cyclic_encode(cyclic_train[:,1:2], 6)
sin_hour, cos_hour = cyclic_encode(cyclic_train[:,2:3], 24)
sin_winter, cos_winter = cyclic_encode(cyclic_train[:,3:4], 4)
sin_spring, cos_spring = cyclic_encode(cyclic_train[:,4:5], 4)
sin_summer, cos_summer = cyclic_encode(cyclic_train[:,5:6], 4)
sin_fall, cos_fall = cyclic_encode(cyclic_train[:,6:7], 4)
sin_year, cos_year = cyclic_encode(cyclic_train[:,7:8], 365)

train_cyclic = np.concatenate((sin_month, cos_month, sin_week, cos_week, sin_hour, cos_hour,
                               sin_winter, cos_winter, sin_spring, cos_spring, sin_summer, cos_summer,
                               sin_fall, cos_fall, sin_year, cos_year), axis=1)

## 8. Combine & Save Final Files

In [10]:
train_final = np.concatenate((scaled_train, train_categorical, train_cyclic), axis=1)

columns = ['aep', 'Is_holiday1','Is_holiday2', 'Is_Weekend1','Is_Weekend2',
           'sin_month', 'cos_month', 'sin_week','cos_week', 'sin_hour', 'cos_hour',
           'sin_winter','cos_winter','sin_spring','cos_spring','sin_summer','cos_summer',
           'sin_fall','cos_fall','sin_year_day','cos_year_day']

train_df = pd.DataFrame(train_final, columns=columns)
train_df.to_csv('7_AEP_train.csv', index=False)
print("Train file saved successfully!")

Train file saved successfully!
